# SmolLM2 Profiles and Hugging Face Base Weights

DustyLM includes tiny scratch profiles and larger SmolLM2-compatible profiles. The motivation is simple: an 8M model is useful for learning the full training loop, but it cannot carry much broad language knowledge or reasoning. If you want a more capable base model, start from a pretrained 135M or 360M SmolLM2 checkpoint and then use SFT as alignment for your target behavior.

This notebook shows how profiles work, how to download and convert Hugging Face SmolLM2 weights into DustyLM checkpoints, how to generate from the converted base model, and how SFT fits into the same profile-driven workflow.

## Setup

Run this in Colab, RunPod, or a local notebook. It clones the repo, installs DustyLM, and does not download the Dusty datasets.

In [ ]:
# 1. Clone the repo and enter the directory
!git clone https://github.com/mkhordoo/dusty-lm.git
%cd dusty-lm

# 2. Install uv
!pip install -q uv

# 3. Install the project into the notebook environment
!uv pip install --system -e .

## Why Use SmolLM2 as a Base Model?

The `dusty8m` profile learns vocabulary, syntax, world patterns, and personality from the local Dusty data. That is great for education because every step is visible, but it is small enough that capability is limited.

SmolLM2 profiles solve a different problem. Hugging Face already trained these models on a much larger corpus, so the base checkpoint already knows a tokenizer, English token statistics, and broader text patterns. DustyLM converts those HF weights into the local module layout, then generation and SFT use the same profile registry as the tiny model.

Do not retrain the tokenizer for these profiles. The SmolLM2 weights were trained with a specific tokenizer, and the profile points to that tokenizer under `artifacts/tokenizers/smollm2_tokenizer.json`.

## Inspect the Profiles

Profiles are the source of truth for model size, tokenizer, checkpoint paths, HF artifact locations, generation defaults, and SFT training config.

In [ ]:
from dustylm.config import get_profile, list_profiles

for name in ["dusty8m", "smollm2_135m", "smollm2_360m", "sft_smollm2_360m"]:
    profile = get_profile(name)
    model = profile.model
    print("=" * 80)
    print("profile:", profile.name)
    print("family:", model.family)
    print("layers:", model.num_layers)
    print("hidden dim:", model.embed_dim)
    print("heads:", model.num_heads, "query /", model.num_kv_heads, "kv")
    print("vocab:", model.vocab_size)
    print("tokenizer:", model.tokenizer.path_or_name)
    if profile.generation:
        print("checkpoint:", profile.generation.checkpoint_path)
    if profile.hf_artifacts:
        print("hf repo:", profile.hf_artifacts.repo_id)
        print("hf weights cache:", profile.hf_artifacts.local_weights_path)

## Download and Convert a Base Checkpoint

`dustylm.artifacts download --convert` does two things:

1. Downloads Hugging Face `model.safetensors` into `artifacts/hf/` and the tokenizer into `artifacts/tokenizers/`.
2. Converts the HF key names into a DustyLM `.pt` checkpoint under `artifacts/checkpoints/`.

The 135M profile is smaller and better for a first notebook run. The 360M profile is more capable but needs more memory and disk.

In [ ]:
PROFILE = "smollm2_135m"  # Change to "smollm2_360m" when you have enough memory/disk.

!uv run python -m dustylm.artifacts download --profile {PROFILE} --convert

## Confirm Artifact Paths

After conversion, generation reads the `.pt` checkpoint from `artifacts/checkpoints/`. There should be no need for a top-level `checkpoints/` directory.

In [ ]:
from pathlib import Path
from dustylm.config import get_profile

profile = get_profile(PROFILE)
paths = [
    profile.hf_artifacts.local_weights_path,
    profile.hf_artifacts.local_tokenizer_path,
    profile.generation.checkpoint_path,
]

for path in paths:
    path = Path(path)
    print(path, "OK" if path.exists() else "MISSING")

## Generate From the Pretrained Base Model

This is not Dusty-aligned yet. It is the general SmolLM2 base model running through DustyLM's local generation stack.

In [ ]:
!uv run python -m dustylm.generate \
  --profile {PROFILE} \
  --prompt "A robot vacuum found a crumb under the couch and" \
  --temperature 0.8 \
  --top-p 0.9


## How SFT Fits In

SFT is the alignment step: the pretrained SmolLM2 base already has broad language knowledge, and SFT teaches it the target response style or task behavior. The training mechanics are the same profile-driven path used elsewhere in DustyLM; the profile decides which tokenizer, dataset, initialization checkpoint, output checkpoint, and context length to use.

For SmolLM2 profiles, do not run `make dusty-tokenizer`. The tokenizer belongs to the pretrained model and is downloaded from Hugging Face. Your SFT dataset still needs to be tokenized/prepared for the SFT profile before training.

The current repository includes `sft_smollm2_360m` as the SmolLM2 SFT profile. For a custom Dusty-style SmolLM2 run, define or adjust an SFT profile so it points at your SFT JSONL/tokenized dataset and initializes from the converted base checkpoint.

In [ ]:
# Example commands for a SmolLM2 SFT run after your SFT profile and dataset are ready.
# These are intentionally commented out because the dataset/profile choice is project-specific.

# !uv run python -m dustylm.data_prep --profile sft_smollm2_360m
# !uv run python -m dustylm.train --profile sft_smollm2_360m --epochs 1 --checkpoint-every-steps 100
# !uv run python -m dustylm.inference --profile sft_smollm2_360m

## Practical Guidance

- Use `dusty8m` when you want to learn the full stack from scratch.
- Use `smollm2_135m` when you want a more capable pretrained base that is still relatively manageable.
- Use `smollm2_360m` when you have enough GPU memory and want stronger base capability.
- Keep SmolLM2 artifacts under `artifacts/hf/`, `artifacts/tokenizers/`, and `artifacts/checkpoints/`.
- Keep profile names and checkpoint paths explicit; that makes switching base models and SFT targets a config change instead of a code change.